# CV Masterclass 02: Architectural Evolution

Welcome to Notebook 02. Knowing how a convolution works is the first step. Knowing how to stack 152 convolutions on top of each other without destroying the network is the second step.

We will explore the paradigm shift from building networks 'Wider and Shallower' to 'Narrower and Deeper', culminating in Kaiming He's Nobel-worthy ResNet architecture.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin. The answer involves mathematical degradation:

> *"A $1\times1$ convolution seems physically useless—it just multiplies a pixel by a number. Why are they the most critical computational component in deep architectures like Inception and ResNet?"*

---

1. **The VGG Era:** The power of depth and the parameter explosion.
2. **The Inception Module:** Multi-Scale Parallel Processing and the $1\times1$ Bottleneck.
3. **The Degradation Problem:** Why 56 layers performs worse than 20 layers.
4. **Residual Networks (ResNet):** Skip Connections and Bottlenecks.
5. **MobileNetV2:** Inverted Residuals & Linear Bottlenecks.
6. **Squeeze-and-Excitation (SENet):** Channel attention mechanisms.
7. **DenseNet:** All-to-All Skip Connections and Mem checks.
8. **EfficientNet & Compound Scaling:** The empirical sizing law.
9. **ConvNeXt:** Modernizing the CNN for the Transformer Era.
10. **Modern Normalization & Regularization:** LayerNorm and Stochastic Depth (DropPath).
11. **Structural Reparameterization (RepVGG):** The Training-Inference decoupling.
12. **Core Concepts & Pitfalls Summary.**


## 📑 Table of Contents

* [🎓 Socratic Deep Check](#socratic-deep-check)
* [1. The VGG Era & The Parameter Explosion](#1-the-vgg-era-the-parameter-explosion)
* [2. The Inception Module (Multi-Scale Parallel Processing)](#2-the-inception-module-multi-scale-parallel-processing)
  * [The Solution: The $1\times1$ Bottleneck Prefix](#the-solution-the-1times1-bottleneck-prefix)
* [3. The Degradation Problem](#3-the-degradation-problem)
* [4. Residual Networks (ResNet)](#4-residual-networks-resnet)
* [5. MobileNetV2: Inverted Residuals](#5-mobilenetv2-inverted-residuals)
* [6. Squeeze-and-Excitation (SENet)](#6-squeeze-and-excitation-senet)
* [7. DenseNet (Concatenation Logic)](#7-densenet-concatenation-logic)
* [8. EfficientNet & Compound Scaling](#8-efficientnet-compound-scaling)
* [9. ConvNeXt: Modernizing the CNN](#9-convnext-modernizing-the-cnn)
* [10. Modern Normalization & Regularization](#10-modern-normalization-regularization)
  * [10.1 LayerNorm vs BatchNorm](#101-layernorm-vs-batchnorm)
  * [10.2 Stochastic Depth (DropPath)](#102-stochastic-depth-droppath)
* [11. Structural Reparameterization (RepVGG)](#11-structural-reparameterization-repvgg)
* [12. Core Concepts & Pitfalls Summary](#12-core-concepts-pitfalls-summary)
  * [❌ PITFALL 1: Ignoring Quadratic Growth in DenseNets](#pitfall-1-ignoring-quadratic-growth-in-densenets)
  * [❌ PITFALL 2: Misplacing ReLU in Shortcuts](#pitfall-2-misplacing-relu-in-shortcuts)
  * [❌ PITFALL 3: Manifold Collapse in MobileNetV2](#pitfall-3-manifold-collapse-in-mobilenetv2)


### 🎓 Junior to Senior: Concept Bridge

**Junior:** Sees VGG, ResNet, and EfficientNet as black boxes that just 'get better scores'. Defaults to ResNet50 for everything.

**Senior:** Tracks the explicit constraints each architecture solved. Knows VGG proved depth matters, ResNet solved the vanishing gradient via skip connections, and EfficientNet mathematically balanced width, depth, and resolution for edge deployment.

---


### 🎓 Junior to Senior: Concept Bridge

**Junior:** Sees VGG, ResNet, and EfficientNet as black boxes that just 'get better scores'. Defaults to ResNet50 for everything.

**Senior:** Tracks the explicit constraints each architecture solved. Knows VGG proved depth matters, ResNet solved the vanishing gradient via skip connections, and EfficientNet mathematically balanced width, depth, and resolution for edge deployment.

---


### 🎓 Junior to Senior: Concept Bridge

**Junior:** Sees VGG, ResNet, and EfficientNet as black boxes that just 'get better scores'. Defaults to ResNet50 for everything.

**Senior:** Tracks the explicit constraints each architecture solved. Knows VGG proved depth matters, ResNet solved the vanishing gradient via skip connections, and EfficientNet mathematically balanced width, depth, and resolution for edge deployment.

---


## 1. The VGG Era & The Parameter Explosion

In 2014, VGG-16 formalized: Always use $3\times3$ convolutions stacked. Never use massive $5\times5$ or $7\times7$ single layers.

**The Cost:** $3 \times 3 \times 512 \times 512 = 2.3 \text{ Million}$ parameters for a *single layer*. This caused massive memory caching issues and computational overhead.


## 2. The Inception Module (Multi-Scale Parallel Processing)

GoogLeNet applied $1\times1$, $3\times3$, AND $5\times5$ convolutions in parallel. 

### The Solution: The $1\times1$ Bottleneck Prefix
A $1\times1$ convolution performs no spatial filtering. It only acts across the **channel dimension** to perform cheap cross-channel dimensionality reduction (compression) before expensive $3\times3$ spatial filtering.


In [ ]:
import torch
import torch.nn as nn

class InceptionModule(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.branch1 = nn.Conv2d(in_channels, 64, kernel_size=1)
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, 48, kernel_size=1),
            nn.Conv2d(48, 128, kernel_size=3, padding=1)
        )
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=1),
            nn.Conv2d(16, 32, kernel_size=5, padding=2)
        )
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, 32, kernel_size=1)
        )

    def forward(self, x):
        out1 = self.branch1(x)
        out2 = self.branch2(x)
        out3 = self.branch3(x)
        out4 = self.branch4(x)
        return torch.cat([out1, out2, out3, out4], 1)

dummy_in = torch.randn(2, 192, 28, 28)
inception = InceptionModule(192)
print(f"Inception Output Shape: {inception(dummy_in).shape}")


## 3. The Degradation Problem

A 56-layer network often has *higher* training error than a 20-layer network. This is not overfitting; it's **optimization failure**. Deep non-linear layers struggle to learn a simple Identity Mapping ($f(x)=x$) because weight matrices are biased toward non-zero transformations.


## 4. Residual Networks (ResNet)

**Skip Connection:** We explicitly calculate $out = F(x) + x$. 
If a layer is useless, the network easily learns weights such that $F(x) \rightarrow 0$. The output becomes exactly $x$ (Identity). This allows gradients to flow through an unbroken "superhighway" during backprop.


In [ ]:
class ResNetBottleneck(nn.Module):
    def __init__(self, in_channels, bottleneck_channels, out_channels, stride=1):
        super().__init__()
        self.residual = nn.Sequential(
            nn.Conv2d(in_channels, bottleneck_channels, 1, bias=False),
            nn.BatchNorm2d(bottleneck_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(bottleneck_channels, bottleneck_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(bottleneck_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(bottleneck_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        )
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.residual(x) + self.shortcut(x))


## 5. MobileNetV2: Inverted Residuals

Standard bottlenecks go **Wide → Narrow → Wide**. 
**Inverted Residuals** go **Narrow → Wide → Narrow**. Why? Depthwise convolutions are more expressive in high dimensions. We expand first, do depthwise, then compress to a **Linear Bottleneck** (no ReLU after the final $1\times1$) to prevent information manifold collapse.

## 6. Squeeze-and-Excitation (SENet)

**Channel Attention:** Not all feature channels are equal. SENet globally pools channels into scalars, passes them through a tiny 2-layer MLP to learn channel relationships, and re-weights the original channels using a Sigmoid importance score.

## 7. DenseNet (Concatenation Logic)

Instead of adding $F(x) + x$, DenseNet **concatenates** all previous layers' outputs. Every layer receives the explicit information of all predecessors: $L_{n}(x) = F(x_n, [x_0, x_1, ..., x_{n-1}])$.

## 8. EfficientNet & Compound Scaling

Empirically, scaling only Depth, Width, or Resolution is inefficient. EfficientNet scales all three using a unified coefficient $\phi$ based on the law: $\alpha \cdot \beta^2 \cdot \gamma^2 \approx 2$.

## 9. ConvNeXt: Modernizing the CNN

ConvNeXt applies Transformer design choices to ResNets: larger kernels (7x7), **LayerNorm** instead of BatchNorm, and **GELU** instead of ReLU. It matches ViT performance while remaining a pure CNN.

## 10. Modern Normalization & Regularization

### 10.1 LayerNorm vs BatchNorm
- **BatchNorm**: Normalizes across the batch. Fast but unstable on small batches.
- **LayerNorm**: Normalizes across the channel/feature dimension per instance. Essential for Transformer-style vision backbones.

### 10.2 Stochastic Depth (DropPath)
Instead of dropping neurons, we drop **entire residual blocks** during training with probability $p$. This forces the network to be robust to missing computation paths.

In [ ]:
def drop_path(x, drop_prob: float = 0.0, training: bool = False):
    """
    Stochastic Depth (DropPath) Implementation.
    Basically, set entire sample outputs to 0.0 with probability drop_prob.
    """
    if drop_prob == 0. or not training:
        return x
    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)  # (B, 1, 1, 1)
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    binary_mask = torch.floor(random_tensor)
    output = x.div(keep_prob) * binary_mask
    return output

class ModernBlock(nn.Module):
    def __init__(self, dim, drop_path_prob=0.1):
        super().__init__()
        self.conv = nn.Conv2d(dim, dim, 3, padding=1)
        self.drop_path = drop_path_prob
        
    def forward(self, x):
        # Residual connection with Stochastic Depth
        return x + drop_path(self.conv(x), self.drop_path, self.training)

dummy = torch.randn(4, 16, 224, 224)
block = ModernBlock(16, drop_path_prob=0.5)
print(f"Stochastic Depth Output: {block(dummy).shape}")


## 11. Structural Reparameterization (RepVGG)

RepVGG uses multi-branch paths during training (3x3 + 1x1 + Identity) but **collapses** them into a single 3x3 convolution during inference using **BN-Folding** math. This gives you ResNet-level accuracy with VGG-level speed.

In [ ]:
def fuse_bn_tensor(conv_w, bn_rm, bn_rv, bn_eps, bn_gamma, bn_beta):
    std = (bn_rv + bn_eps).sqrt()
    t = (bn_gamma / std).reshape(-1, 1, 1, 1)
    return conv_w * t, bn_beta - bn_rm * bn_gamma / std

print("Reference Section 11 implementation for BN-Folding and Kernel Fusion math.")

---
## ✅ Knowledge Check

### Q1: What specific problem did ResNet's residual connections solve?
<details><summary>👉 Answer</summary>

They prevented the vanishing gradient problem in extremely deep networks by allowing gradients to flow directly backwards through the addition operations.
</details>

### Q2: How does Depthwise Separable Convolution drastically reduce parameter counts in MobileNets?
<details><summary>👉 Answer</summary>

It splits the convolution into two steps: a spatial convolution per channel (depthwise) followed by a 1x1 cross-channel convolution (pointwise), reducing mults from `K*K*C` to `K*K + C`.
</details>

---
## 🔨 Practical Practice

### Exercise 1
Skip Connections: Implement a ResNet bottleneck building block manually in PyTorch.

### Exercise 2
Scaling Laws: Calculate the computational cost ratio between a standard CNN and a MobileNet building block for a 256-channel input.


## 12. Core Concepts & Pitfalls Summary

### ❌ PITFALL 1: Ignoring Quadratic Growth in DenseNets
Memory balloons quadratically. Use **Gradient Checkpointing** to trade compute for VRAM.

### ❌ PITFALL 2: Misplacing ReLU in Shortcuts
Always apply the Shortcut ReLU *after* addition. Pre-activation ResNet avoids this by moving activation inside the residual path entirely.

### ❌ PITFALL 3: Manifold Collapse in MobileNetV2
NEVER place a ReLU after the final $1\times1$ expansion-in-bottleneck layer. It kills information in the compressed manifold.